# Comparació d'Algorismes RL: DQN vs NFSP vs PPO

**Condicions d'entrenament uniformes:**
- Observació plana de **239 dims**
- Xarxa **[256, 256]** hidden layers per a tots tres
- **48 entorns paral·lels** (`SubprocVecEnv`), **24M timesteps**
- Mateixos oponents: 5% Random – 65% AgentRegles – 30% Self-play
- `puntuacio_final = 24`
- Mètrica d'avaluació: `0.25 × WR_random + 0.75 × WR_regles`

**Run analitzat:** `resultats_comparativa_28_03_1800h`


In [ ]:
import os, sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


In [ ]:
# Estil global dels gràfics
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.size'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

COLORS = {'dqn': '#e74c3c', 'nfsp': '#2ecc71', 'ppo': '#3498db'}
LABELS = {'dqn': 'DQN', 'nfsp': 'NFSP', 'ppo': 'PPO'}

# Localitzar directori de resultats
NOTEBOOK_DIR = Path(os.getcwd())
BASE_DIR = None
for root in [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent]:
    candidates = sorted(root.glob('resultats_comparativa_*'))
    if candidates:
        BASE_DIR = candidates[-1]
        break

if BASE_DIR is None:
    raise FileNotFoundError("No s'ha trobat cap directori resultats_comparativa_*")

# Afegir arrel del projecte al path Python
PROJECT_ROOT = BASE_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Directori de resultats: {BASE_DIR}")
print(f"Project root:           {PROJECT_ROOT}")


## 1. Càrrega dels logs d'entrenament


In [ ]:
logs = {}
for algo in ['dqn', 'nfsp', 'ppo']:
    path = BASE_DIR / algo / 'training_log.csv'
    if path.exists():
        df = pd.read_csv(path)
        logs[algo] = df
        best_m    = df['eval_metric'].max()
        best_step = df.loc[df['eval_metric'].idxmax(), 'step']
        final_m   = df['eval_metric'].iloc[-1]
        elapsed_h = df['elapsed_s'].iloc[-1] / 3600
        print(f"{algo.upper():5s} | {len(df):3d} evals | "
              f"Millor mètrica: {best_m:5.1f}% (step {best_step/1e6:.1f}M) | "
              f"Final: {final_m:5.1f}% | "
              f"Temps: {elapsed_h:.2f}h")
    else:
        print(f"AVÍS: no trobat {path}")

print(f"\nAlgorismes carregats: {list(logs.keys())}")


## 2. Corbes d'aprenentatge


In [ ]:
def smooth(series, w=3):
    return series.rolling(window=w, min_periods=1, center=True).mean()


metrics_plot = [
    ('eval_wr_regles', 'Win Rate vs AgentRegles (%)'),
    ('eval_wr_random', 'Win Rate vs Random (%)'),
    ('eval_metric',    'Mètrica combinada (%)\n0.25×Random + 0.75×Regles'),
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, (col, ylabel) in zip(axes, metrics_plot):
    for algo, df in logs.items():
        if col in df.columns:
            x = df['step'] / 1e6
            ax.plot(x, df[col],         alpha=0.2, color=COLORS[algo], linewidth=0.8)
            ax.plot(x, smooth(df[col]), alpha=0.9, color=COLORS[algo], linewidth=2.2,
                    label=LABELS[algo])
    ax.set_xlabel('Timesteps (M)')
    ax.set_ylabel(ylabel.split('\n')[0])
    ax.set_title(ylabel.split('\n')[0], fontsize=11)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.25)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
    ax.set_xlim(left=0)

plt.suptitle("Corbes d'aprenentatge – DQN vs NFSP vs PPO (24M timesteps)",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(BASE_DIR / 'corbes_aprenentatge.png', bbox_inches='tight', dpi=150)
plt.show()


## 3. Temps d'entrenament i eficiència de mostreig


In [ ]:
# Llegir resum_temps.txt (fallback als elapsed_s dels logs)
resum_path = BASE_DIR / 'resum_temps.txt'
times_s = {}
if resum_path.exists():
    for line in resum_path.read_text().splitlines():
        for algo in ['dqn', 'nfsp', 'ppo']:
            if line.lower().startswith(algo + ':'):
                try:
                    secs = int(line.split(':')[1].strip().split('s')[0])
                    times_s[algo.upper()] = secs
                except Exception:
                    pass

if not times_s:
    for algo, df in logs.items():
        times_s[algo.upper()] = df['elapsed_s'].iloc[-1]

algos_ord = ['DQN', 'NFSP', 'PPO']
hores = [times_s.get(a, 0) / 3600 for a in algos_ord]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Temps total
ax = axes[0]
bars = ax.bar(algos_ord, hores,
              color=[COLORS[a.lower()] for a in algos_ord], width=0.5, edgecolor='white')
ax.bar_label(bars, labels=[f'{h:.2f}h' for h in hores], padding=4, fontsize=12, fontweight='bold')
ax.set_ylabel("Hores d'entrenament")
ax.set_title("Temps total d'entrenament", fontweight='bold')
ax.grid(True, axis='y', alpha=0.25)
ax.set_ylim(0, max(hores) * 1.25)

# Eficiència: timesteps per hora
ax = axes[1]
total_steps = 24e6
msteps_h = [total_steps / (t * 1e6) if t > 0 else 0 for t in hores]
bars2 = ax.bar(algos_ord, msteps_h,
               color=[COLORS[a.lower()] for a in algos_ord], width=0.5, edgecolor='white')
ax.bar_label(bars2, labels=[f'{v:.1f}' for v in msteps_h], padding=4, fontsize=12, fontweight='bold')
ax.set_ylabel('M timesteps / hora')
ax.set_title('Eficiència de mostreig', fontweight='bold')
ax.grid(True, axis='y', alpha=0.25)

plt.suptitle("Temps i eficiència d'entrenament", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(BASE_DIR / 'temps_entrenament.png', bbox_inches='tight', dpi=150)
plt.show()


## 4. Taula de rendiment final


In [ ]:
rows = []
for algo, df in logs.items():
    best_idx = df['eval_metric'].idxmax()
    rows.append({
        'Algorisme':        algo.upper(),
        'WR Random (màx)':  round(df['eval_wr_random'].max(), 1),
        'WR Regles (màx)':  round(df['eval_wr_regles'].max(), 1),
        'Mètrica (màx)':    round(df['eval_metric'].max(), 1),
        'Mètrica (final)':  round(df['eval_metric'].iloc[-1], 1),
        'Step màx (M)':     round(df.loc[best_idx, 'step'] / 1e6, 1),
        'Temps (h)':        round(df['elapsed_s'].iloc[-1] / 3600, 2),
    })

df_resum = pd.DataFrame(rows).set_index('Algorisme')
print(df_resum.to_string())


## 5. Evolució detallada de la mètrica combinada


In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

for algo, df in logs.items():
    x    = df['step'] / 1e6
    y_sm = smooth(df['eval_metric'], w=4)
    ax.plot(x, df['eval_metric'], alpha=0.18, color=COLORS[algo], linewidth=0.8)
    ax.plot(x, y_sm,              alpha=0.95, color=COLORS[algo], linewidth=2.5,
            label=LABELS[algo])
    bi = df['eval_metric'].idxmax()
    ax.scatter(df.loc[bi, 'step'] / 1e6, df.loc[bi, 'eval_metric'],
               color=COLORS[algo], s=120, zorder=5, marker='*')

ax.set_xlabel('Timesteps (M)')
ax.set_ylabel('Mètrica combinada (%)')
ax.set_title('Evolució de la mètrica combinada (suavitzada)', fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.25)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))
ax.set_xlim(left=0)

plt.tight_layout()
plt.savefig(BASE_DIR / 'metrica_combinada.png', bbox_inches='tight', dpi=150)
plt.show()


## 6. Mini-lliga: enfrontaments directes (head-to-head)

Carreguem els millors models (`best.pt`) de cada algorisme i els enfrontem en **500 partides per combinació** (250 amb cada posició de taula).


In [ ]:
import torch
import torch.nn as nn

from rlcard.agents import DQNAgent, NFSPAgent
from joc.entorn import TrucEnv
from joc.entorn.cartes_accions import ACTION_LIST
from RL.models.model_propi.agent_regles import AgentRegles
from RL.entrenament.entrenamentsComparatius.entrenament_comparatiu import (
    SimpleActorCritic, SimpleActorCriticAgent, wrap_env_aplanat
)

N_ACTIONS = len(ACTION_LIST)
OBS_DIM   = 239
HIDDEN    = [256, 256]
device    = torch.device('cpu')

ENV_H2H = {'num_jugadors': 2, 'cartes_jugador': 3, 'puntuacio_final': 24, 'seed': 999}


In [ ]:
def carregar_dqn(path):
    agent = DQNAgent(
        num_actions=N_ACTIONS, state_shape=OBS_DIM, mlp_layers=HIDDEN,
        learning_rate=1e-4, batch_size=256, replay_memory_size=1000,
        replay_memory_init_size=100, update_target_estimator_every=99999,
        epsilon_decay_steps=1, epsilon_end=0.0, train_every=99999, device=device,
    )
    sd = torch.load(path, map_location=device, weights_only=True)
    agent.q_estimator.qnet.load_state_dict(sd)
    return agent


def carregar_nfsp(path):
    agent = NFSPAgent(
        num_actions=N_ACTIONS, state_shape=OBS_DIM,
        hidden_layers_sizes=HIDDEN, q_mlp_layers=HIDDEN,
        rl_learning_rate=1e-4, sl_learning_rate=1e-4, batch_size=256,
        reservoir_buffer_capacity=1000, q_replay_memory_size=1000,
        q_update_target_estimator_every=99999, anticipatory_param=0.25,
        q_epsilon_decay_steps=1, q_epsilon_end=0.0,
        q_replay_memory_init_size=100, q_batch_size=256,
        q_train_every=99999, device=device,
    )
    sd = torch.load(path, map_location=device, weights_only=True)
    agent._rl_agent.q_estimator.qnet.load_state_dict(sd['q_net'])
    agent.policy_network.load_state_dict(sd['sl_net'])
    return agent


def carregar_ppo(path):
    net = SimpleActorCritic(OBS_DIM, N_ACTIONS, 256)
    net.load_state_dict(torch.load(path, map_location=device, weights_only=True))
    return SimpleActorCriticAgent(net, N_ACTIONS, device)


LOADERS = {'dqn': carregar_dqn, 'nfsp': carregar_nfsp, 'ppo': carregar_ppo}


In [ ]:
models = {}
for algo, loader in LOADERS.items():
    p = BASE_DIR / algo / 'best.pt'
    if p.exists():
        models[algo] = loader(str(p))
        print(f"{algo.upper()}: model carregat correctament")
    else:
        print(f"AVÍS: no trobat {p}")

regles_agent = AgentRegles(num_actions=N_ACTIONS, seed=777)
all_agents = {**models, 'regles': regles_agent}
print(f"\nAgents disponibles: {list(all_agents.keys())}")


In [ ]:
def head_to_head(agent_a, agent_b, n_per_side=500, seed=999):
    """Enfronta agent_a vs agent_b en 2*n_per_side partides totals.

    Cada agent juga n_per_side partides com a jugador 0 (local)
    i n_per_side com a jugador 1 (visitant).
    Retorna (wr_a, wr_b, wr_ties) en percentatge.
    """
    env = wrap_env_aplanat(TrucEnv({**ENV_H2H, 'seed': seed}))
    wins_a = wins_b = ties = 0
    total  = 2 * n_per_side

    for agents, idx_a in [([agent_a, agent_b], 0), ([agent_b, agent_a], 1)]:
        env.set_agents(agents)
        for _ in range(n_per_side):
            _, payoffs = env.run(is_training=False)
            if   payoffs[idx_a] > 0: wins_a += 1
            elif payoffs[idx_a] < 0: wins_b += 1
            else:                    ties   += 1

    return 100.0*wins_a/total, 100.0*wins_b/total, 100.0*ties/total


In [ ]:
N_PER_SIDE = 500
N_GAMES    = N_PER_SIDE * 2

print(f"Executant mini-lliga: {N_PER_SIDE}×2 = {N_GAMES} partides per combinació...\n")

pairs = [
    ('ppo',  'dqn'),
    ('ppo',  'nfsp'),
    ('dqn',  'nfsp'),
    ('ppo',  'regles'),
    ('dqn',  'regles'),
    ('nfsp', 'regles'),
]

h2h = {}
for a, b in pairs:
    if a in all_agents and b in all_agents:
        wr_a, wr_b, _ = head_to_head(all_agents[a], all_agents[b], N_PER_SIDE)
        h2h[(a, b)] = wr_a
        print(f"  {a.upper():6s} vs {b.upper():6s}: {wr_a:.1f}% – {wr_b:.1f}%")
    else:
        missing = a if a not in all_agents else b
        print(f"  SKIP: {missing} no disponible")

print("\nMini-lliga completada.")


In [ ]:
algo_keys = [a for a in ['ppo', 'dqn', 'nfsp', 'regles'] if a in all_agents]
n = len(algo_keys)
matrix = np.full((n, n), np.nan)

for i, a in enumerate(algo_keys):
    for j, b in enumerate(algo_keys):
        if i == j:
            matrix[i, j] = 50.0
        elif (a, b) in h2h:
            matrix[i, j] = h2h[(a, b)]
        elif (b, a) in h2h:
            matrix[i, j] = 100.0 - h2h[(b, a)]

fig, ax = plt.subplots(figsize=(7, 6))
im   = ax.imshow(matrix, cmap='RdYlGn', vmin=20, vmax=80)
cbar = plt.colorbar(im, ax=ax, label='Win Rate (%)', shrink=0.85)
cbar.ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

tick_labels = [a.upper() for a in algo_keys]
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(tick_labels, fontsize=12, fontweight='bold')
ax.set_yticklabels(tick_labels, fontsize=12, fontweight='bold')
ax.set_xlabel('Oponent', fontsize=12)
ax.set_ylabel('Agent', fontsize=12)
ax.set_title(f"Mini-lliga Head-to-Head ({N_GAMES} partides/combinació)\n"
             "Win rate de la fila contra la columna",
             fontsize=11, fontweight='bold')

for i in range(n):
    for j in range(n):
        if not np.isnan(matrix[i, j]):
            val   = matrix[i, j]
            color = 'white' if (val < 33 or val > 67) else 'black'
            ax.text(j, i, f'{val:.1f}%', ha='center', va='center',
                    fontsize=13, fontweight='bold', color=color)

plt.tight_layout()
plt.savefig(BASE_DIR / 'head_to_head.png', bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
points = {a: 0 for a in algo_keys}
wins_c = {a: 0 for a in algo_keys}
loss_c = {a: 0 for a in algo_keys}

for (a, b), wr_a in h2h.items():
    if a not in algo_keys or b not in algo_keys:
        continue
    wr_b = 100 - wr_a
    if wr_a > wr_b:
        points[a] += 3; wins_c[a] += 1; loss_c[b] += 1
    elif wr_b > wr_a:
        points[b] += 3; wins_c[b] += 1; loss_c[a] += 1
    else:
        points[a] += 1; points[b] += 1

ranking = sorted(algo_keys, key=lambda x: points[x], reverse=True)

print('RÀNQUING MINI-LLIGA')
print('=' * 36)
print(f"{'Pos':4s} {'Agent':8s} {'Pts':5s} {'V':5s} {'D':5s}")
print('-' * 36)
for pos, algo in enumerate(ranking, 1):
    print(f"{pos:<4d} {algo.upper():<8s} {points[algo]:<5d} {wins_c[algo]:<5d} {loss_c[algo]:<5d}")


## 7. Panell resum visual


In [ ]:
fig = plt.figure(figsize=(16, 11))

# 1. Corba mètrica combinada (suavitzada + marcador del millor punt)
ax1 = fig.add_subplot(2, 2, 1)
for algo, df in logs.items():
    y_sm = smooth(df['eval_metric'], w=4)
    ax1.plot(df['step']/1e6, y_sm, color=COLORS[algo], linewidth=2.2, label=LABELS[algo])
    bi = df['eval_metric'].idxmax()
    ax1.scatter(df.loc[bi, 'step']/1e6, df.loc[bi, 'eval_metric'],
                color=COLORS[algo], s=90, zorder=5, marker='*')
ax1.set_xlabel('Timesteps (M)'); ax1.set_ylabel('Mètrica (%)')
ax1.set_title("Mètrica combinada d'aprenentatge", fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.25)
ax1.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

# 2. Win rate vs AgentRegles
ax2 = fig.add_subplot(2, 2, 2)
for algo, df in logs.items():
    y_sm = smooth(df['eval_wr_regles'], w=4)
    ax2.plot(df['step']/1e6, y_sm, color=COLORS[algo], linewidth=2.2, label=LABELS[algo])
ax2.set_xlabel('Timesteps (M)'); ax2.set_ylabel('WR vs AgentRegles (%)')
ax2.set_title('Win Rate vs AgentRegles (oponent principal)', fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.25)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

# 3. Temps total d'entrenament
ax3 = fig.add_subplot(2, 2, 3)
bars3 = ax3.bar(algos_ord, hores,
                color=[COLORS[a.lower()] for a in algos_ord], width=0.5, edgecolor='white')
ax3.bar_label(bars3, labels=[f'{h:.2f}h' for h in hores], padding=4, fontweight='bold')
ax3.set_ylabel('Hores'); ax3.set_title("Temps total d'entrenament", fontweight='bold')
ax3.grid(True, axis='y', alpha=0.25); ax3.set_ylim(0, max(hores)*1.25)

# 4. Eficiència: millor mètrica vs temps
ax4 = fig.add_subplot(2, 2, 4)
for algo, df in logs.items():
    bm = df['eval_metric'].max()
    th = df['elapsed_s'].iloc[-1] / 3600
    ax4.scatter(th, bm, color=COLORS[algo], s=280, zorder=5,
                marker='o', edgecolors='white', linewidths=2)
    ax4.annotate(algo.upper(), (th, bm), textcoords='offset points',
                 xytext=(8, 5), fontsize=13, fontweight='bold', color=COLORS[algo])
ax4.set_xlabel('Temps entrenament (h)'); ax4.set_ylabel('Millor mètrica (%)')
ax4.set_title('Eficiència: rendiment vs cost computacional', fontweight='bold')
ax4.grid(True, alpha=0.25)
ax4.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f%%'))

plt.suptitle('Panell Resum – Comparativa DQN / NFSP / PPO',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(BASE_DIR / 'panel_resum.png', bbox_inches='tight', dpi=150)
plt.show()


## 8. Conclusions

### Un resultat sorprenent

L'anàlisi revela dos aspectes que aparentment es contradiuen, però que junts expliquen completament el comportament dels tres algorismes.

---

### 8.1 Resultats quantitatius

| | DQN | NFSP | PPO |
|---|---|---|---|
| **Millor mètrica** | 43.2% (step 0.5M!) | 30.0% (step 1.0M) | **51.8%** (step 10M) |
| **Mètrica final** | 19.2% ↘ | 7.2% ↘↘ | **38.5%** → |
| **Temps entrenament** | 0.91h | 3.60h | **0.65h** |
| **Tendència** | Col·lapse | Col·lapse sever | Estable |

**Head-to-head (best.pt de cada, 500 partides/parell):**

| | vs DQN | vs NFSP | vs PPO | vs Regles |
|---|---|---|---|---|
| **DQN** | — | **79.4%** | **74.6%** | 34.0% |
| **NFSP** | 20.6% | — | 31.0% | 12.4% |
| **PPO** | 25.4% | **69.0%** | — | 38.0% |
| **Regles** | **65.6%** | **87.0%** | **62.0%** | — |

---

### 8.2 Per interpretar el rànquing correctament

El resultat de DQN > PPO en el head-to-head **no significa que DQN sigui un millor algorisme d'aprenentatge**. Cal entendre el context:

- El `best.pt` de DQN va ser guardat en el **step 0.5M** — quan la seva mètrica era 43.2% (86% vs random, 33% vs regles).
- El `best.pt` de PPO es va guardar al **step 10M** (mètrica 51.8%) — un model molt més equilibrat però que DQN de step 0.5M pot explotar.
- La corba de DQN **col·lapsa** completament després del step 2M.

En resum: **DQN aprèn ràpid i oblida; PPO aprèn lent però consolida.**

### 8.3 Per què col·lapsa DQN?

- **Off-policy + 48 envs paral·lels**: el replay buffer omple amb transicions de 48 polítiques simultànies.
- **Una sola passada d'actualització**: DQN fa 1 gradient step per cada 32 transicions noves.
- **Oblit catastròfic**: la xarxa Q es sobreescriu constantment i no consolida el que aprèn.

### 8.4 Per què és molt pitjor NFSP?

- **5.5× més lent que PPO** (3.60h vs 0.65h) sense cap benefici de qualitat.
- Mètrica final 7.2%, perd **87%** de les partides contra AgentRegles.
- NFSP busca equilibri de Nash en jocs simètrics (póquer). Al Truc amb oponents heterogenis, aquest equilibri no existeix.

### 8.5 Per què AgentRegles guanya a tothom?

Cap dels tres agents RL aconsegueix superar l'agent de regles heurístic. 24M timesteps sense un extractor de característiques especialitzat (COS) no és suficient per aprendre les estratègies subtils del Truc.

### 8.6 Conclusió i recomanació

> **PPO és l'algorisme d'aprenentatge més robust i eficient** dels tres per a aquest escenari: corba estable, temps mínim i millor mètrica objectiva (51.8%).

> La combinació **PPO + COS** és la millor estratègia global per a agents competitius al Truc.


In [ ]:
print('=' * 60)
print('RESUM EXECUTIU FINAL')
print('=' * 60)
print(f'Run: {BASE_DIR.name}')
print()

for algo, df in logs.items():
    bm   = df['eval_metric'].max()
    fm   = df['eval_metric'].iloc[-1]
    th   = df['elapsed_s'].iloc[-1] / 3600
    jocs = int(df['games_played'].iloc[-1])
    print(f'{algo.upper()}: millor={bm:.1f}%  final={fm:.1f}%  temps={th:.2f}h  jocs={jocs:,}')

if h2h:
    print()
    print(f'Head-to-head ({N_GAMES} partides/combinació):')
    for (a, b), wr_a in h2h.items():
        guanya = a.upper() if wr_a > 50 else b.upper()
        print(f'  {a.upper()} vs {b.upper()}: {wr_a:.1f}%–{100-wr_a:.1f}%  → guanya {guanya}')

print()
print('Rànquing final:')
for pos, algo in enumerate(ranking, 1):
    print(f'  {pos}. {algo.upper()}  ({points[algo]} pts, {wins_c[algo]}V-{loss_c[algo]}D)')
